In [0]:
!pip install xgboost shap

In [0]:
# ────────────────────────────────────────────────
# WEEK 5 — Explainable AI
# claim_denial_model → Feature Importance → Reasons → gold_claim_explanations
# ────────────────────────────────────────────────

# CELL 1 — Imports
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd
from pyspark.sql import Window
from pyspark.sql.functions import dense_rank, col, when

CATALOG = "main"
SCHEMA  = "gold"

In [0]:
# CELL 2 — Load gold_claim_features
gold_claim_features = spark.table(f"{CATALOG}.{SCHEMA}.gold_claim_features")
print("Row count:", gold_claim_features.count())
gold_claim_features.show(3)

In [0]:
# CELL 3 — Recreate encoding (same logic as training)
cat_cols = ["specialty", "category", "location"]

ml_df = gold_claim_features.fillna({
    "diagnosis_count"    : 0,
    "provider_risk_score": 0.0,
    "billed_vs_avg_cost" : 0.0,
    "claim_frequency"    : 0
})

for c in cat_cols:
    window = Window.orderBy(c)
    ml_df = ml_df.withColumn(c + "_idx", dense_rank().over(window) - 1)

In [0]:
# CELL 4 — Load registered XGBoost model from Unity Catalog
mlflow.set_registry_uri("databricks-uc")

model_uri = f"models:/{CATALOG}.{SCHEMA}.claim_denial_model/1"
xgb_model = mlflow.xgboost.load_model(model_uri)

print("Model loaded:", type(xgb_model))

In [0]:
# CELL 5 — Extract feature importance
feature_cols = [
    "billed_amount", "billed_vs_avg_cost", "high_cost_flag",
    "severity_score", "specialty_idx", "category_idx", "location_idx"
]

importance_scores = xgb_model.feature_importances_

importance_df = pd.DataFrame({
    "feature"   : feature_cols,
    "importance": importance_scores
}).sort_values("importance", ascending=False)

print(importance_df)

In [0]:
# CELL 6 — Feature to business reason mapping
FEATURE_REASON_MAP = {
    "billed_amount"     : "Billed amount is unusually high",
    "billed_vs_avg_cost": "Billed amount significantly exceeds average cost",
    "high_cost_flag"    : "Claim is flagged as high cost",
    "severity_score"    : "Severity level of the condition is high",
    "specialty_idx"     : "Specialty type is associated with higher denial rate",
    "category_idx"      : "Claim category has elevated risk",
    "location_idx"      : "Provider location shows elevated risk pattern"
}

In [0]:
# CELL 7 — Score all claims and generate explanations
feature_cols = [
    "billed_amount", "billed_vs_avg_cost", "high_cost_flag",
    "severity_score", "specialty_idx", "category_idx", "location_idx"
]

# Convert to pandas for XGBoost
claims_pd = ml_df.select(["claim_id"] + feature_cols).toPandas()

X = claims_pd[feature_cols].values
probs      = xgb_model.predict_proba(X)[:, 1]
predictions = xgb_model.predict(X)

claims_pd["risk_score"]  = probs
claims_pd["risk_label"]  = np.where(probs >= 0.5, "HIGH", "LOW")
claims_pd["prediction"]  = np.where(predictions == 1, "DENIED", "APPROVED")

In [0]:
# CELL 8 — Get top 2 reasons per claim using SHAP values (correct method)
import shap

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X)

# shap_values shape: (n_claims, n_features)
# Each value = how much that feature pushed the prediction toward denial

def get_top_reasons_shap(shap_row, top_n=2):
    # Use absolute SHAP value to find most impactful features
    scored      = {feature_cols[i]: abs(shap_row[i]) for i in range(len(feature_cols))}
    top_features = sorted(scored, key=scored.get, reverse=True)[:top_n]
    return [FEATURE_REASON_MAP[f] for f in top_features]

claims_pd["reasons"] = [get_top_reasons_shap(shap_values[i]) for i in range(len(claims_pd))]
claims_pd["reason_1"] = claims_pd["reasons"].apply(lambda x: x[0] if len(x) > 0 else None)
claims_pd["reason_2"] = claims_pd["reasons"].apply(lambda x: x[1] if len(x) > 1 else None)

In [0]:
# CELL 9 — Build final explanation table and save to Delta
explanation_df = claims_pd[[
    "claim_id", "risk_score", "risk_label", "prediction", "reason_1", "reason_2"
]]

explanation_spark = spark.createDataFrame(explanation_df)

explanation_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_claim_explanations")

print("Saved gold_claim_explanations")
explanation_spark.show(10, truncate=False)

In [0]:
# CELL 10 — Verify
spark.table(f"{CATALOG}.{SCHEMA}.gold_claim_explanations").show(10, truncate=False)